# 01. Data Preparation

3개 신용 데이터셋의 로딩, 불필요 항목 삭제, 결측치 처리를 수행한다.

| Dataset | 출처 | 샘플 수 | Target |
|---|---|---|---|
| GMSC | Give Me Some Credit (Kaggle) | ~150K | SeriousDlqin2yrs (0/1) |
| HELOC | FICO Explainable ML Challenge | ~10K | RiskPerformance (Good/Bad) |
| HC | Home Credit Default Risk (Kaggle) | ~307K | TARGET (0/1) |

In [1]:
import numpy as np
import pandas as pd

DATA_DIR = '../.data'

---
## 1. GMSC (Give Me Some Credit)

- 전체 숫자형
- 결측: `MonthlyIncome`, `NumberOfDependents`에 존재
- 처리: 중앙값 대체

In [2]:
gmsc_raw = pd.read_csv(f'{DATA_DIR}/give_me_some_credit/cs-training.csv', index_col=0)
print(f'shape: {gmsc_raw.shape}')
gmsc_raw.head()

shape: (150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [3]:
# 결측 현황
miss = gmsc_raw.isnull().sum()
miss[miss > 0]

MonthlyIncome         29731
NumberOfDependents     3924
dtype: int64

In [4]:
# Target 분리
y_gmsc = gmsc_raw['SeriousDlqin2yrs']
X_gmsc = gmsc_raw.drop('SeriousDlqin2yrs', axis=1)

# 결측치: 중앙값 대체
X_gmsc = X_gmsc.fillna(X_gmsc.median())

print(f'X: {X_gmsc.shape}, y: {y_gmsc.shape}')
print(f'default rate: {y_gmsc.mean():.2%}')
print(f'remaining missing: {X_gmsc.isnull().sum().sum()}')

X: (150000, 10), y: (150000,)
default rate: 6.68%
remaining missing: 0


---
## 2. HELOC (Home Equity Line of Credit)

- Target: `RiskPerformance` (Good/Bad) → 1(Bad) / 0(Good)
- 특수값 결측 처리:
  - `-9`: No Bureau Record or No Investigation → 결측 처리
  - `-8`: No Usable/Valid Trades or Inquiries → 결측 처리
  - `-7`: Condition not Met (e.g., No Delinquency) → 유효값으로 유지

In [5]:
heloc_raw = pd.read_csv(f'{DATA_DIR}/heloc_dataset/heloc_dataset_v1.txt')
print(f'shape: {heloc_raw.shape}')
heloc_raw.head()

shape: (10459, 24)


,RiskPerformance,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,...,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
0,Bad,55,144,4,84,20,3,0,83,2,...,43,0,0,0,33,-8,8,1,1,69
1,Bad,61,58,15,41,2,4,4,100,-7,...,67,0,0,0,0,-8,0,-8,-8,0
2,Bad,67,66,5,24,9,0,0,100,-7,...,44,0,4,4,53,66,4,2,1,86
3,Bad,66,169,1,73,28,1,1,93,76,...,57,0,5,4,72,83,6,4,3,91
4,Bad,81,333,27,132,12,0,0,100,-7,...,25,0,1,1,51,89,3,1,0,80


In [6]:
# Target 분리 (Bad=1, Good=0)
y_heloc = (heloc_raw['RiskPerformance'] == 'Bad').astype(int)
X_heloc = heloc_raw.drop('RiskPerformance', axis=1).copy()

print(f'target distribution:\n{y_heloc.value_counts()}')
print(f'bad rate: {y_heloc.mean():.2%}')

target distribution:
RiskPerformance
1    5459
0    5000
Name: count, dtype: int64
bad rate: 52.19%


In [7]:
# 특수값 확인
special_counts = pd.DataFrame({
    '-9 (No Record)': (X_heloc == -9).sum(),
    '-8 (No Trades)': (X_heloc == -8).sum(),
    '-7 (Cond. Not Met)': (X_heloc == -7).sum(),
})
special_counts[special_counts.sum(axis=1) > 0]

,-9 (No Record),-8 (No Trades),-7 (Cond. Not Met)
ExternalRiskEstimate,598,0,0
MSinceOldestTradeOpen,588,239,0
MSinceMostRecentTradeOpen,588,0,0
AverageMInFile,588,0,0
NumSatisfactoryTrades,588,0,0
NumTrades60Ever2DerogPubRec,588,0,0
NumTrades90Ever2DerogPubRec,588,0,0
PercentTradesNeverDelq,588,0,0
MSinceMostRecentDelq,588,176,4664
MaxDelq2PublicRecLast12M,588,0,0


In [8]:
# -9, -8 → NaN (정보 부재). -7은 유효값으로 유지.
X_heloc = X_heloc.replace([-9, -8], np.nan)

# 결측 비율 확인
miss = X_heloc.isnull().mean()
print('결측 비율:')
print(miss[miss > 0].sort_values(ascending=False).map('{:.1%}'.format))

결측 비율:
NetFractionInstallBurden              38.3%
NumInstallTradesWBalance              13.9%
NumBank2NatlTradesWHighUtilization    11.2%
MSinceMostRecentInqexcl7days          10.2%
MSinceOldestTradeOpen                  7.9%
NetFractionRevolvingBurden             7.4%
MSinceMostRecentDelq                   7.3%
NumRevolvingTradesWBalance             7.1%
PercentTradesWBalance                  5.8%
ExternalRiskEstimate                   5.7%
NumTrades90Ever2DerogPubRec            5.6%
MSinceMostRecentTradeOpen              5.6%
NumTrades60Ever2DerogPubRec            5.6%
NumSatisfactoryTrades                  5.6%
AverageMInFile                         5.6%
PercentInstallTrades                   5.6%
NumTradesOpeninLast12M                 5.6%
NumTotalTrades                         5.6%
MaxDelqEver                            5.6%
PercentTradesNeverDelq                 5.6%
MaxDelq2PublicRecLast12M               5.6%
NumInqLast6M                           5.6%
NumInqLast6Mexcl7days    

In [9]:
# 결측치: 중앙값 대체
X_heloc = X_heloc.fillna(X_heloc.median())

print(f'X: {X_heloc.shape}, y: {y_heloc.shape}')
print(f'remaining missing: {X_heloc.isnull().sum().sum()}')

X: (10459, 23), y: (10459,)
remaining missing: 0


---
## 3. HC (Home Credit)

- 122개 컬럼 중 숫자형만 사용
- `SK_ID_CURR` (고객 ID) 제거
- 결측 40% 이상 변수 제거
- 나머지 결측: 중앙값 대체

In [10]:
hc_raw = pd.read_csv(f'{DATA_DIR}/home_credit/application_train.csv')
print(f'shape: {hc_raw.shape}')
hc_raw.head()

shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
# Target 분리, ID 제거
y_hc = hc_raw['TARGET']
X_hc = hc_raw.drop(columns=['TARGET', 'SK_ID_CURR'])

# 숫자형만
X_hc = X_hc.select_dtypes(include=[np.number])
print(f'숫자형 변수: {X_hc.shape[1]}개')
print(f'default rate: {y_hc.mean():.2%}')

숫자형 변수: 104개
default rate: 8.07%


In [12]:
# 결측 40% 이상 변수 제거
miss_ratio = X_hc.isnull().mean()
high_miss = miss_ratio[miss_ratio >= 0.4]
print(f'결측 40% 이상: {len(high_miss)}개 변수 제거')
print(high_miss.sort_values(ascending=False).head(10).map('{:.1%}'.format))

X_hc = X_hc[X_hc.columns[miss_ratio < 0.4]]
print(f'\n남은 변수: {X_hc.shape[1]}개')

결측 40% 이상: 45개 변수 제거
COMMONAREA_AVG              69.9%
COMMONAREA_MEDI             69.9%
COMMONAREA_MODE             69.9%
NONLIVINGAPARTMENTS_AVG     69.4%
NONLIVINGAPARTMENTS_MODE    69.4%
NONLIVINGAPARTMENTS_MEDI    69.4%
LIVINGAPARTMENTS_AVG        68.4%
LIVINGAPARTMENTS_MEDI       68.4%
LIVINGAPARTMENTS_MODE       68.4%
FLOORSMIN_MODE              67.8%
dtype: str

남은 변수: 59개


In [13]:
# FLAG 변수 중 소수 클래스 1% 미만인 이진 변수 제거
flag_cols = [c for c in X_hc.columns if 'FLAG' in c]
drop_flags = []
for col in flag_cols:
    vc = X_hc[col].value_counts(normalize=True)
    if len(vc) == 2 and vc.min() < 0.01:
        drop_flags.append(col)

X_hc = X_hc.drop(columns=drop_flags)
print(f'FLAG 변수 중 소수 클래스 <1%: {len(drop_flags)}개 제거')
print(f'  제거: {drop_flags}')
print(f'  남은 변수: {X_hc.shape[1]}개')

FLAG 변수 중 소수 클래스 <1%: 18개 제거
  제거: ['FLAG_MOBIL', 'FLAG_CONT_MOBILE', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21']
  남은 변수: 41개


In [14]:
# 나머지 결측: 중앙값 대체
X_hc = X_hc.fillna(X_hc.median())

print(f'X: {X_hc.shape}, y: {y_hc.shape}')
print(f'remaining missing: {X_hc.isnull().sum().sum()}')

X: (307511, 41), y: (307511,)


remaining missing: 0


---
## Summary

In [15]:
summary = pd.DataFrame({
    'Dataset': ['GMSC', 'HELOC', 'HC'],
    'Samples': [len(X_gmsc), len(X_heloc), len(X_hc)],
    'Features': [X_gmsc.shape[1], X_heloc.shape[1], X_hc.shape[1]],
    'Default Rate': [y_gmsc.mean(), y_heloc.mean(), y_hc.mean()],
})
summary['Default Rate'] = summary['Default Rate'].map('{:.2%}'.format)
summary

,Dataset,Samples,Features,Default Rate
0,GMSC,150000,10,6.68%
1,HELOC,10459,23,52.19%
2,HC,307511,41,8.07%


---
## Save

In [16]:
import pickle, os

prepared = {
    'GMSC': {'X': X_gmsc, 'y': y_gmsc},
    'HELOC': {'X': X_heloc, 'y': y_heloc},
    'HC': {'X': X_hc, 'y': y_hc},
}

out_path = f'{DATA_DIR}/prepared.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(prepared, f)

size_mb = os.path.getsize(out_path) / 1024 / 1024
print(f'Saved to {out_path} ({size_mb:.1f} MB)')

Saved to ../.data/prepared.pkl (113.0 MB)